# PPT Asset Generator

This Colab notebook generates slide-ready tables and figures for the final presentation.

Run all cells from top to bottom. After execution, all outputs will be saved in:

`ppt_assets/`

The generated files are named by slide number, so the presentation team can insert them directly into the PPT.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/hyeon03-sketch/IML-Final-project.git"
BRANCH = "codex/ml-project-review"
REPO_DIR = Path("/content/IML-Final-project")

if Path("/content").exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
else:
    os.chdir(Path.cwd())

print("Working directory:", Path.cwd())
print("Dataset exists:", Path("IML_Final_dataset.xlsx").exists())

required = ["pandas", "numpy", "matplotlib", "seaborn", "sklearn", "xgboost", "shap", "openpyxl"]
missing = []
for package in required:
    try:
        __import__(package)
    except Exception:
        missing.append(package)

if missing:
    install_names = ["scikit-learn" if p == "sklearn" else p for p in missing]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *install_names])
    print("Installed:", install_names)
else:
    print("All required packages are already installed.")


## 1. Imports, Style, and Helper Functions

The following cell sets the presentation style and defines helper functions for saving tables and figures.


In [ ]:
import warnings
from pathlib import Path
import shutil
import textwrap
import subprocess

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as patches
import seaborn as sns
import shap

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RND = 42
DATA_PATH = Path("IML_Final_dataset.xlsx")
SHEET_NAME = "최종데이터셋_모델용"
TARGET = "전세환산보증금(만원)"
CONVERSION_RATE = 0.065
OUT_DIR = Path("ppt_assets")
OUT_DIR.mkdir(exist_ok=True)

PALETTE = {
    "blue": "#4267B2",
    "navy": "#1F2A44",
    "cyan": "#42C8D7",
    "green": "#2E8B57",
    "red": "#C2410C",
    "gray": "#6B7280",
    "light_gray": "#F3F4F6",
    "dark": "#111827",
}

plt.rcParams["figure.dpi"] = 130
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)


def setup_font():
    candidate_paths = [
        Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"),
        Path("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"),
        Path("/System/Library/Fonts/AppleSDGothicNeo.ttc"),
        Path("/Library/Fonts/AppleGothic.ttf"),
    ]
    if not any(p.exists() for p in candidate_paths):
        try:
            subprocess.check_call(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.check_call(["apt-get", "install", "-y", "fonts-nanum"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as exc:
            print("Font installation skipped:", exc)

    for cache_file in Path.home().glob(".cache/matplotlib/fontlist*"):
        try:
            cache_file.unlink()
        except Exception:
            pass

    font_paths = [p for p in candidate_paths if p.exists()]
    for font_path in font_paths:
        try:
            fm.fontManager.addfont(str(font_path))
        except Exception:
            pass
    try:
        fm._load_fontmanager(try_read_cache=False)
    except Exception:
        pass

    for font_path in font_paths:
        try:
            font_name = fm.FontProperties(fname=str(font_path)).get_name()
            mpl.rcParams["font.family"] = [font_name]
            mpl.rcParams["font.sans-serif"] = [font_name]
            mpl.rcParams["axes.unicode_minus"] = False
            sns.set_theme(style="whitegrid", font=font_name)
            print("Font:", font_name)
            return font_name
        except Exception:
            continue
    mpl.rcParams["axes.unicode_minus"] = False
    sns.set_theme(style="whitegrid")
    print("Font: default")
    return "default"


FONT_NAME = setup_font()


def save_current_fig(filename):
    path = OUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", path)
    return path


def wrap_text(value, width=28):
    text = str(value)
    if len(text) <= width:
        return text
    return "\n".join(textwrap.wrap(text, width=width, break_long_words=False))


def save_table(df, filename, title=None, subtitle=None, col_width=24, font_size=12, scale_y=1.4):
    show_df = df.copy()
    show_df = show_df.applymap(lambda x: wrap_text(x, col_width))
    nrows, ncols = show_df.shape
    fig_w = max(8, min(18, ncols * 2.4))
    fig_h = max(2.2, 1.2 + nrows * 0.55)
    if title:
        fig_h += 0.5
    if subtitle:
        fig_h += 0.35

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")

    y = 0.98
    if title:
        ax.text(0.0, y, title, transform=ax.transAxes, ha="left", va="top",
                fontsize=18, fontweight="bold", color=PALETTE["navy"])
        y -= 0.09
    if subtitle:
        ax.text(0.0, y, subtitle, transform=ax.transAxes, ha="left", va="top",
                fontsize=11, color=PALETTE["gray"])
        y -= 0.08

    table = ax.table(
        cellText=show_df.values,
        colLabels=show_df.columns,
        loc="center",
        cellLoc="center",
        colLoc="center",
        bbox=[0, 0, 1, max(0.2, y - 0.03)],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, scale_y)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#D1D5DB")
        cell.set_linewidth(0.8)
        if row == 0:
            cell.set_facecolor(PALETTE["blue"])
            cell.set_text_props(color="white", weight="bold")
        elif row % 2 == 0:
            cell.set_facecolor(PALETTE["light_gray"])
        else:
            cell.set_facecolor("white")

    path = OUT_DIR / filename
    plt.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", path)
    return path


def save_metric_cards(metrics, filename, title):
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 2.3))
    if len(metrics) == 1:
        axes = [axes]
    for ax, (label, value, note) in zip(axes, metrics):
        ax.axis("off")
        card = patches.FancyBboxPatch((0.02, 0.08), 0.96, 0.84, boxstyle="round,pad=0.02,rounding_size=0.03",
                                      linewidth=1.2, edgecolor="#D1D5DB", facecolor="white")
        ax.add_patch(card)
        ax.text(0.08, 0.72, label, transform=ax.transAxes, fontsize=12, color=PALETTE["gray"], weight="bold")
        ax.text(0.08, 0.42, value, transform=ax.transAxes, fontsize=23, color=PALETTE["navy"], weight="bold")
        ax.text(0.08, 0.20, note, transform=ax.transAxes, fontsize=10, color=PALETTE["gray"])
    fig.suptitle(title, fontsize=18, fontweight="bold", color=PALETTE["navy"], x=0.01, ha="left")
    return save_current_fig(filename)


## 2. Load and Prepare the Final Modeling Dataset

This section constructs the final target and checks dong-level spatial variables.


In [ ]:
def make_jeonse_equivalent_target(data, conversion_rate=CONVERSION_RATE):
    out = data.copy()
    deposit = pd.to_numeric(out["보증금(만원)"], errors="coerce")
    monthly_rent = pd.to_numeric(out["월세금(만원)"], errors="coerce").fillna(0)
    lease_type = out["전월세구분"].astype(str).str.strip()
    target = deposit.astype(float).copy()
    monthly_mask = lease_type.eq("월세")
    target.loc[monthly_mask] = deposit.loc[monthly_mask] + monthly_rent.loc[monthly_mask] * 12.0 / conversion_rate
    out[TARGET] = target
    return out


def first_mode(series):
    modes = series.mode(dropna=False)
    return modes.iloc[0] if len(modes) else np.nan


def harmonize_dong_level_features(data, feature_cols, group_col="읍면동"):
    out = data.copy()
    rows = []
    for col in feature_cols:
        unique_by_dong = out.groupby(group_col)[col].nunique(dropna=False)
        inconsistent = unique_by_dong[unique_by_dong > 1]
        before = out[col].copy()
        out[col] = out.groupby(group_col)[col].transform(first_mode)
        changed = int((before.astype(str) != out[col].astype(str)).sum())
        rows.append({
            "Feature": col,
            "Dongs inconsistent before": len(inconsistent),
            "Rows harmonized": changed,
            "Max unique values after": int(out.groupby(group_col)[col].nunique(dropna=False).max()),
        })
    return out, pd.DataFrame(rows)


DONG_LEVEL_COLS = [
    "바다여부", "공원여부", "공원수", "최대공원면적(㎡)", "총공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_총학교수", "동_초중고모두있음여부",
]

raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df = make_jeonse_equivalent_target(raw)
df = df[df[TARGET].notna() & df[TARGET].gt(0)].copy()
df, harmonization_report = harmonize_dong_level_features(df, DONG_LEVEL_COLS)

dataset_summary = pd.DataFrame([
    ["Total observations", f"{len(df):,}"],
    ["Jeonse transactions", f"{int(df['전월세구분'].eq('전세').sum()):,}"],
    ["Monthly-rent transactions converted", f"{int(df['전월세구분'].eq('월세').sum()):,}"],
    ["Number of dongs", f"{df['읍면동'].nunique():,}"],
    ["Target mean", f"{df[TARGET].mean():,.2f} 만원"],
    ["Target median", f"{df[TARGET].median():,.2f} 만원"],
], columns=["Item", "Value"])

display(dataset_summary)
display(harmonization_report)

dataset_summary.to_csv(OUT_DIR / "slide_06_dataset_overview.csv", index=False, encoding="utf-8-sig")
harmonization_report.to_csv(OUT_DIR / "slide_13_dong_harmonization_report.csv", index=False, encoding="utf-8-sig")


## 3. Dataset and Preprocessing Assets

These figures can be used in the dataset and preprocessing slides.


In [ ]:
save_table(
    dataset_summary,
    "slide_06_dataset_overview_table.png",
    title="Final Modeling Dataset",
    subtitle="Buk-gu, Pohang apartment lease transactions"
)

save_metric_cards(
    [
        ("Total observations", f"{len(df):,}", "Final rows used for modeling"),
        ("Jeonse rows", f"{int(df['전월세구분'].eq('전세').sum()):,}", "Original jeonse transactions"),
        ("Converted rows", f"{int(df['전월세구분'].eq('월세').sum()):,}", "Monthly-rent rows converted"),
        ("Dongs", f"{df['읍면동'].nunique():,}", "Dong-level local markets"),
    ],
    "slide_08_dataset_metric_cards.png",
    "Dataset Snapshot"
)

lease_counts = df["전월세구분"].value_counts().reindex(["전세", "월세"]).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(["Jeonse", "Monthly rent\nconverted"], lease_counts.values, color=[PALETTE["blue"], PALETTE["cyan"]])
axes[0].set_title("Transaction Composition", fontsize=15, fontweight="bold", color=PALETTE["navy"])
axes[0].set_ylabel("Number of observations")
for i, v in enumerate(lease_counts.values):
    axes[0].text(i, v + max(lease_counts.values) * 0.02, f"{int(v):,}", ha="center", weight="bold")

axes[1].pie(lease_counts.values, labels=["Jeonse", "Monthly rent converted"], autopct="%1.1f%%",
            startangle=90, colors=[PALETTE["blue"], PALETTE["cyan"]], textprops={"fontsize": 11})
axes[1].set_title("Share of Transaction Types", fontsize=15, fontweight="bold", color=PALETTE["navy"])
save_current_fig("slide_08_transaction_composition.png")

plt.figure(figsize=(10, 5.2))
sns.histplot(df[TARGET], bins=40, kde=True, color=PALETTE["blue"])
plt.axvline(df[TARGET].mean(), color=PALETTE["red"], linestyle="--", linewidth=2, label=f"Mean: {df[TARGET].mean():,.0f}")
plt.axvline(df[TARGET].median(), color=PALETTE["green"], linestyle="-", linewidth=2, label=f"Median: {df[TARGET].median():,.0f}")
plt.title("Distribution of Jeonse-Equivalent Deposit", fontsize=17, fontweight="bold", color=PALETTE["navy"])
plt.xlabel("Jeonse-equivalent deposit (10,000 KRW)")
plt.ylabel("Count")
plt.legend()
save_current_fig("slide_09_target_distribution.png")

formula_df = pd.DataFrame([
    ["Jeonse transaction", "Jeonse-equivalent deposit = deposit"],
    ["Monthly-rent transaction", "Jeonse-equivalent deposit = deposit + monthly rent x 12 / 0.065"],
], columns=["Transaction type", "Target construction"])
save_table(
    formula_df,
    "slide_10_target_formula_table.png",
    title="Target Variable Construction",
    subtitle="The model predicts jeonse-equivalent deposit, not raw deposit.",
    col_width=42
)

save_table(
    harmonization_report,
    "slide_13_dong_harmonization_table.png",
    title="Dong-Level Spatial Variable Consistency Check",
    subtitle="Dong-level variables were harmonized using the within-dong mode.",
    col_width=24,
    font_size=10,
    scale_y=1.25
)


## 4. Dong-Level EDA Assets

These outputs support the claim that dong-level location should be included in the baseline.


In [ ]:
dong_summary = (
    df.groupby("읍면동")
    .agg(
        거래수=(TARGET, "size"),
        평균_전세환산보증금=(TARGET, "mean"),
        중앙값_전세환산보증금=(TARGET, "median"),
        평균_전용면적=("전용면적(㎡)", "mean"),
        평균_건물연령=("건물연령(계약기준)", "mean"),
        평균_층=("층", "mean"),
        바다여부=("바다여부", "first"),
        공원여부=("공원여부", "first"),
        공원수=("공원수", "first"),
        최대공원면적=("최대공원면적(㎡)", "first"),
        초등학교수=("동_초등학교수", "first"),
        중학교수=("동_중학교수", "first"),
        고등학교수=("동_고등학교수", "first"),
        초중고모두있음=("동_초중고모두있음여부", "first"),
    )
    .reset_index()
    .sort_values("평균_전세환산보증금", ascending=False)
)
dong_summary.to_csv(OUT_DIR / "slide_18_dong_level_summary.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(10, 8))
plot_df = dong_summary.sort_values("평균_전세환산보증금", ascending=True)
colors = [PALETTE["cyan"] if v else PALETTE["blue"] for v in plot_df["바다여부"]]
plt.barh(plot_df["읍면동"], plot_df["평균_전세환산보증금"], color=colors)
plt.title("Average Jeonse-Equivalent Deposit by Dong", fontsize=17, fontweight="bold", color=PALETTE["navy"])
plt.xlabel("Average jeonse-equivalent deposit (10,000 KRW)")
plt.ylabel("Dong")
plt.figtext(0.02, 0.01, "Cyan bars indicate sea-proximity dongs.", fontsize=10, color=PALETTE["gray"])
save_current_fig("slide_18_dong_mean_target_bar.png")

corr_cols = [
    "평균_전세환산보증금", "거래수", "평균_전용면적", "평균_건물연령", "평균_층",
    "바다여부", "공원여부", "공원수", "최대공원면적",
    "초등학교수", "중학교수", "고등학교수", "초중고모두있음",
]
corr = dong_summary[corr_cols].corr(numeric_only=True)
corr.to_csv(OUT_DIR / "slide_18_dong_level_correlation.csv", encoding="utf-8-sig")

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, linewidths=0.5, cbar_kws={"shrink": 0.75})
plt.title("Dong-Level Correlation Heatmap", fontsize=17, fontweight="bold", color=PALETTE["navy"])
save_current_fig("slide_18_dong_correlation_heatmap.png")


## 5. Feature Design and Multicollinearity Assets

The final design includes dong in the baseline and removes redundant aggregate spatial variables.


In [ ]:
BASE_FEATURES = ["전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월"]
DONG_FEATURE = ["읍면동"]
SPATIAL_FEATURES = [
    "바다여부",
    "공원여부",
    "공원수",
    "최대공원면적(㎡)",
    "동_초등학교수",
    "동_중학교수",
    "동_고등학교수",
    "동_초중고모두있음여부",
]

FEATURE_GROUPS = {
    "A_location_baseline": BASE_FEATURES + DONG_FEATURE,
    "B_location_spatial": BASE_FEATURES + DONG_FEATURE + SPATIAL_FEATURES,
}

BINARY_COLS = {"바다여부", "공원여부", "동_초중고모두있음여부"}
CATEGORICAL_COLS = {"읍면동"}

feature_set_table = pd.DataFrame([
    ["A_location_baseline", "Baseline with location control", "exclusive area, floor, building age, contract year, contract month, dong"],
    ["B_location_spatial", "Spatial-extended model", "A + sea proximity, park variables, school-count variables"],
], columns=["Experiment", "Purpose", "Included variables"])
save_table(
    feature_set_table,
    "slide_14_feature_set_design_table.png",
    title="Feature Set Design",
    subtitle="Dong is included in the baseline as a location control.",
    col_width=44,
    font_size=11,
    scale_y=1.5
)

spatial_category_table = pd.DataFrame([
    ["Sea", "바다여부", "Coastal or near-coastal dong dummy"],
    ["Park", "공원여부, 공원수, 최대공원면적(㎡)", "Green-space availability and scale"],
    ["School", "동_초등학교수, 동_중학교수, 동_고등학교수", "Educational infrastructure by dong"],
    ["Education mix", "동_초중고모두있음여부", "Whether all three school levels are present"],
], columns=["Category", "Variables", "Interpretation"])
save_table(
    spatial_category_table,
    "slide_12_spatial_variable_categories_table.png",
    title="Spatial-Derived Variables",
    subtitle="Area-level variables used to represent local neighborhood context.",
    col_width=35,
    font_size=11,
    scale_y=1.45
)


def calculate_vif(data, cols):
    X = data[cols].apply(pd.to_numeric, errors="coerce").copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    rows = []
    for col in cols:
        y = X[col].values
        others = [c for c in cols if c != col]
        if len(others) == 0 or np.nanstd(y) == 0:
            vif = np.inf
        else:
            model = LinearRegression()
            model.fit(X[others].values, y)
            r2 = model.score(X[others].values, y)
            if r2 >= 0.999999:
                vif = np.inf
            else:
                vif = 1.0 / (1.0 - r2)
        rows.append({"Feature": col, "VIF": vif})
    out = pd.DataFrame(rows).sort_values("VIF", ascending=False)
    out["VIF_display"] = out["VIF"].apply(lambda v: "inf" if np.isinf(v) else f"{v:.2f}")
    return out


candidate_vif_cols = BASE_FEATURES + [
    "바다여부", "공원여부", "공원수", "최대공원면적(㎡)", "총공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_총학교수", "동_초중고모두있음여부",
]
vif_candidates = calculate_vif(df, candidate_vif_cols)
vif_candidates.to_csv(OUT_DIR / "slide_15_vif_candidates.csv", index=False, encoding="utf-8-sig")

vif_table = vif_candidates[["Feature", "VIF_display"]].rename(columns={"VIF_display": "VIF"}).head(12)
save_table(
    vif_table,
    "slide_15_vif_redundancy_check_table.png",
    title="Redundancy Check Before Final Feature Selection",
    subtitle="Redundant aggregate variables were excluded from the final spatial model.",
    col_width=30,
    font_size=11,
    scale_y=1.25
)

feature_decision_table = pd.DataFrame([
    ["총공원면적(㎡)", "Excluded", "High overlap with park-count and maximum-park-area variables"],
    ["동_총학교수", "Excluded", "Aggregate of elementary, middle, and high school counts"],
    ["동_초등학교수 / 동_중학교수 / 동_고등학교수", "Kept", "Retains school-level information separately"],
    ["동_초중고모두있음여부", "Kept", "Captures whether the dong has a full school-level mix"],
], columns=["Variable", "Final decision", "Reason"])
save_table(
    feature_decision_table,
    "slide_16_feature_selection_decisions_table.png",
    title="Final Spatial Feature Decisions",
    subtitle="The final B model keeps interpretable spatial variables and removes redundant aggregates.",
    col_width=38,
    font_size=10.5,
    scale_y=1.45
)


## 6. Model Design Assets

These outputs explain the modeling pipeline and evaluation metrics.


In [ ]:
def save_pipeline_figure():
    steps = [
        ("Raw lease\ntransactions", "MOLIT data"),
        ("Target\nconstruction", "Jeonse-equivalent deposit"),
        ("Feature\nsets", "A baseline vs B spatial"),
        ("XGBoost\nGridSearchCV", "5-fold CV"),
        ("Evaluation", "RMSE, MAE, MAPE, R²"),
        ("Interpretation", "SHAP and holdouts"),
    ]
    fig, ax = plt.subplots(figsize=(14, 3.8))
    ax.axis("off")
    x_positions = np.linspace(0.06, 0.94, len(steps))
    y = 0.55
    for i, ((title, subtitle), x) in enumerate(zip(steps, x_positions)):
        box = patches.FancyBboxPatch((x - 0.07, y - 0.18), 0.14, 0.36,
                                     boxstyle="round,pad=0.02,rounding_size=0.02",
                                     linewidth=1.4, edgecolor=PALETTE["blue"], facecolor="white")
        ax.add_patch(box)
        ax.text(x, y + 0.05, title, ha="center", va="center", fontsize=11.5, weight="bold", color=PALETTE["navy"])
        ax.text(x, y - 0.09, subtitle, ha="center", va="center", fontsize=9.5, color=PALETTE["gray"])
        if i < len(steps) - 1:
            ax.annotate("", xy=(x_positions[i + 1] - 0.085, y), xytext=(x + 0.085, y),
                        arrowprops=dict(arrowstyle="->", lw=1.8, color=PALETTE["gray"]))
    ax.text(0.01, 0.95, "Modeling Pipeline", transform=ax.transAxes, fontsize=18,
            weight="bold", color=PALETTE["navy"], ha="left", va="top")
    return save_current_fig("slide_18_modeling_pipeline.png")


save_pipeline_figure()

implementation_table = pd.DataFrame([
    ["Final model", "XGBoost Regressor"],
    ["Train/test split", "80/20 random split"],
    ["Hyperparameter tuning", "GridSearchCV"],
    ["Cross-validation", "5-fold CV"],
    ["Objective", "Squared error regression"],
    ["Scaler", "Not applied; XGBoost is tree-based"],
], columns=["Item", "Setting"])
save_table(
    implementation_table,
    "slide_18_model_implementation_table.png",
    title="Model Implementation",
    subtitle="The final presentation uses one model for a consistent argument.",
    col_width=42,
    font_size=12,
    scale_y=1.35
)

metrics_table = pd.DataFrame([
    ["RMSE", "Penalizes large prediction errors", "10,000 KRW"],
    ["MAE", "Average absolute prediction error", "10,000 KRW"],
    ["MAPE", "Relative error compared with actual price", "%"],
    ["R²", "Share of variance explained by the model", "0 to 1"],
], columns=["Metric", "Role in evaluation", "Unit"])
save_table(
    metrics_table,
    "slide_19_evaluation_metrics_table.png",
    title="Evaluation Metrics",
    subtitle="MAPE is included to report relative error across price levels.",
    col_width=36,
    font_size=11.5,
    scale_y=1.35
)


## 7. Train XGBoost and Generate Main Result Assets

This section runs the final A/B comparison and saves the result tables.


In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def split_feature_types(cols):
    categorical = [c for c in cols if c in CATEGORICAL_COLS]
    binary = [c for c in cols if c in BINARY_COLS]
    numeric = [c for c in cols if c not in set(categorical + binary)]
    return numeric, categorical, binary


def make_preprocessor(cols):
    numeric, categorical, binary = split_feature_types(cols)
    transformers = []
    pass_cols = numeric + binary
    if pass_cols:
        transformers.append(("pass", "passthrough", pass_cols))
    if categorical:
        transformers.append(("cat", make_one_hot_encoder(), categorical))
    return ColumnTransformer(transformers=transformers, remainder="drop")


def make_xgboost():
    return XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RND,
        n_jobs=-1,
        importance_type="gain",
    )


XGB_PARAM_GRID = {
    "model__n_estimators": [300, 600],
    "model__max_depth": [4, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.9],
    "model__colsample_bytree": [0.9],
}


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape_pct(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def evaluate_predictions(y_true, pred):
    return {
        "test_rmse": rmse(y_true, pred),
        "test_mae": float(mean_absolute_error(y_true, pred)),
        "test_mape_pct": mape_pct(y_true, pred),
        "test_r2": float(r2_score(y_true, pred)),
    }


def fit_xgb_grid(data, group_name, cols, train_idx, test_idx):
    X = data[cols].copy()
    y = data[TARGET].copy()
    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    pipe = Pipeline([("pre", make_preprocessor(cols)), ("model", make_xgboost())])
    search = GridSearchCV(
        pipe,
        param_grid=XGB_PARAM_GRID,
        scoring="neg_root_mean_squared_error",
        cv=KFold(n_splits=5, shuffle=True, random_state=RND),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    pred = search.predict(X_test)
    metrics = evaluate_predictions(y_test, pred)
    metrics.update({
        "experiment": group_name,
        "model": "XGBoost",
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "cv_rmse": float(-search.best_score_),
        "best_params": search.best_params_,
    })
    fitted = {"estimator": search.best_estimator_, "X_test": X_test, "y_test": y_test, "pred": pred}
    return metrics, fitted


train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=RND)

rows = []
fitted = {}
for group_name, cols in FEATURE_GROUPS.items():
    print("Running", group_name)
    metrics, bundle = fit_xgb_grid(df, group_name, cols, train_idx, test_idx)
    rows.append(metrics)
    fitted[group_name] = bundle
    print(f"  RMSE={metrics['test_rmse']:.2f}, MAE={metrics['test_mae']:.2f}, MAPE={metrics['test_mape_pct']:.2f}%, R2={metrics['test_r2']:.4f}")

results = pd.DataFrame(rows).sort_values("test_rmse")
display(results[["experiment", "model", "n_train", "n_test", "cv_rmse", "test_rmse", "test_mae", "test_mape_pct", "test_r2", "best_params"]])
results.to_csv(OUT_DIR / "slide_20_main_xgboost_results.csv", index=False, encoding="utf-8-sig")

a = results[results["experiment"].eq("A_location_baseline")].iloc[0]
b = results[results["experiment"].eq("B_location_spatial")].iloc[0]
main_delta = pd.DataFrame([{
    "Comparison": "B_location_spatial - A_location_baseline",
    "RMSE change": b["test_rmse"] - a["test_rmse"],
    "MAE change": b["test_mae"] - a["test_mae"],
    "MAPE change (%p)": b["test_mape_pct"] - a["test_mape_pct"],
    "R² change": b["test_r2"] - a["test_r2"],
}])
display(main_delta)
main_delta.to_csv(OUT_DIR / "slide_22_main_xgboost_delta.csv", index=False, encoding="utf-8-sig")


In [ ]:
results_display = results.copy()
results_display = results_display[["experiment", "test_rmse", "test_mae", "test_mape_pct", "test_r2"]]
results_display.columns = ["Experiment", "RMSE", "MAE", "MAPE", "R²"]
for col in ["RMSE", "MAE"]:
    results_display[col] = results_display[col].map(lambda x: f"{x:,.2f}")
results_display["MAPE"] = results_display["MAPE"].map(lambda x: f"{x:.2f}%")
results_display["R²"] = results_display["R²"].map(lambda x: f"{x:.4f}")
save_table(
    results_display,
    "slide_20_main_result_table.png",
    title="Main XGBoost Result",
    subtitle="Spatial variables improve all metrics, but the random-split gain is small.",
    col_width=26,
    font_size=11.5,
    scale_y=1.35
)

delta_display = main_delta.copy()
for col in ["RMSE change", "MAE change"]:
    delta_display[col] = delta_display[col].map(lambda x: f"{x:,.2f}")
delta_display["MAPE change (%p)"] = delta_display["MAPE change (%p)"].map(lambda x: f"{x:.3f}")
delta_display["R² change"] = delta_display["R² change"].map(lambda x: f"{x:.4f}")
save_table(
    delta_display,
    "slide_22_main_delta_table.png",
    title="Main Performance Change",
    subtitle="Change is computed as B_location_spatial minus A_location_baseline.",
    col_width=28,
    font_size=11,
    scale_y=1.35
)

plot_metrics = ["test_rmse", "test_mae", "test_mape_pct", "test_r2"]
plot_titles = ["RMSE", "MAE", "MAPE (%)", "R²"]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, metric, title in zip(axes, plot_metrics, plot_titles):
    tmp = results.sort_values("experiment")
    ax.bar(tmp["experiment"], tmp[metric], color=[PALETTE["blue"], PALETTE["cyan"]])
    ax.set_title(title, fontsize=13, fontweight="bold", color=PALETTE["navy"])
    ax.tick_params(axis="x", rotation=35)
    for i, v in enumerate(tmp[metric]):
        label = f"{v:.4f}" if metric == "test_r2" else f"{v:.2f}"
        ax.text(i, v, label, ha="center", va="bottom", fontsize=9)
fig.suptitle("Main Result: Baseline vs Spatial Model", fontsize=17, fontweight="bold", color=PALETTE["navy"])
save_current_fig("slide_20_main_metrics_comparison.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.4))
for ax, group_name in zip(axes, ["A_location_baseline", "B_location_spatial"]):
    bundle = fitted[group_name]
    ax.scatter(bundle["y_test"], bundle["pred"], s=14, alpha=0.45, color=PALETTE["blue"])
    lim = max(bundle["y_test"].max(), np.max(bundle["pred"]))
    ax.plot([0, lim], [0, lim], color=PALETTE["red"], linestyle="--", lw=1.5)
    ax.set_title(group_name, fontsize=13, fontweight="bold", color=PALETTE["navy"])
    ax.set_xlabel("Actual jeonse-equivalent deposit")
    ax.set_ylabel("Predicted")
fig.suptitle("Prediction vs Actual", fontsize=17, fontweight="bold", color=PALETTE["navy"])
save_current_fig("slide_21_prediction_vs_actual.png")


## 8. Strict Holdout Validation Assets

These validation settings are the strongest evidence for the final argument.


In [ ]:
validation_design_table = pd.DataFrame([
    ["Random split", "Random 80/20 split", "Basic model performance"],
    ["Time holdout", "Latest contract year as test set", "Generalization to a newer period"],
    ["Apartment-complex holdout", "Unseen apartment complexes as test set", "Generalization to unseen complexes"],
    ["Dong holdout", "Unseen dongs as test set", "Generalization to unseen local areas"],
], columns=["Validation", "Split design", "Purpose"])
save_table(
    validation_design_table,
    "slide_24_validation_design_table.png",
    title="Additional Holdout Validation",
    subtitle="These tests check whether the result holds beyond a simple random split.",
    col_width=38,
    font_size=11,
    scale_y=1.4
)


def run_holdout(validation_name, train_idx, test_idx):
    rows = []
    for group_name, cols in FEATURE_GROUPS.items():
        print("Strict", validation_name, group_name)
        metrics, _ = fit_xgb_grid(df, group_name, cols, train_idx, test_idx)
        metrics["validation"] = validation_name
        rows.append(metrics)
    return rows


strict_rows = []
train_idx_s, test_idx_s = train_test_split(df.index, test_size=0.2, random_state=RND)
strict_rows.extend(run_holdout("random_split", train_idx_s, test_idx_s))

years = sorted(df["계약연도"].dropna().unique())
if len(years) >= 2:
    latest_year = years[-1]
    train_idx_s = df.index[df["계약연도"] < latest_year]
    test_idx_s = df.index[df["계약연도"] == latest_year]
    strict_rows.extend(run_holdout(f"time_holdout_test_{latest_year}", train_idx_s, test_idx_s))

complex_groups = df["단지명"].fillna("missing_complex")
train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=complex_groups))
strict_rows.extend(run_holdout("complex_holdout", df.index[train_pos], df.index[test_pos]))

dong_groups = df["읍면동"].fillna("missing_dong")
train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=dong_groups))
strict_rows.extend(run_holdout("dong_holdout", df.index[train_pos], df.index[test_pos]))

strict_results = pd.DataFrame(strict_rows).sort_values(["validation", "experiment"])
display(strict_results[["validation", "experiment", "n_train", "n_test", "cv_rmse", "test_rmse", "test_mae", "test_mape_pct", "test_r2", "best_params"]])
strict_results.to_csv(OUT_DIR / "slide_25_strict_validation_results.csv", index=False, encoding="utf-8-sig")

delta_rows = []
for validation in strict_results["validation"].unique():
    sub = strict_results[strict_results["validation"].eq(validation)]
    a = sub[sub["experiment"].eq("A_location_baseline")].iloc[0]
    b = sub[sub["experiment"].eq("B_location_spatial")].iloc[0]
    delta_rows.append({
        "Validation": validation,
        "RMSE change": b["test_rmse"] - a["test_rmse"],
        "MAE change": b["test_mae"] - a["test_mae"],
        "MAPE change (%p)": b["test_mape_pct"] - a["test_mape_pct"],
        "R² change": b["test_r2"] - a["test_r2"],
    })
strict_delta = pd.DataFrame(delta_rows)
display(strict_delta)
strict_delta.to_csv(OUT_DIR / "slide_25_strict_validation_delta.csv", index=False, encoding="utf-8-sig")


In [ ]:
strict_delta_display = strict_delta.copy()
for col in ["RMSE change", "MAE change"]:
    strict_delta_display[col] = strict_delta_display[col].map(lambda x: f"{x:,.2f}")
strict_delta_display["MAPE change (%p)"] = strict_delta_display["MAPE change (%p)"].map(lambda x: f"{x:.3f}")
strict_delta_display["R² change"] = strict_delta_display["R² change"].map(lambda x: f"{x:.4f}")
save_table(
    strict_delta_display,
    "slide_25_holdout_delta_table.png",
    title="Holdout Validation: Spatial Model Change",
    subtitle="Negative RMSE, MAE, and MAPE changes indicate improvement.",
    col_width=30,
    font_size=10.8,
    scale_y=1.3
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, metric, title in zip(axes, ["RMSE change", "MAPE change (%p)", "R² change"], ["RMSE Change", "MAPE Change", "R² Change"]):
    vals = strict_delta[metric].values
    colors = [PALETTE["green"] if (v < 0 and metric != "R² change") or (v > 0 and metric == "R² change") else PALETTE["red"] for v in vals]
    ax.bar(strict_delta["Validation"], vals, color=colors)
    ax.axhline(0, color=PALETTE["dark"], linewidth=1)
    ax.set_title(title, fontsize=13, fontweight="bold", color=PALETTE["navy"])
    ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.3f}" if abs(v) < 10 else f"{v:.1f}", ha="center",
                va="bottom" if v >= 0 else "top", fontsize=9)
fig.suptitle("Spatial Model Improvement Across Holdout Tests", fontsize=17, fontweight="bold", color=PALETTE["navy"])
save_current_fig("slide_25_holdout_delta_bars.png")


## 9. SHAP Interpretation Assets

These outputs support the interpretation slides.


In [ ]:
def classify_feature_group(feature_name):
    clean = feature_name.replace("pass__", "").replace("cat__", "")
    if clean.startswith("읍면동_"):
        return "Dong location"
    if any(x in clean for x in ["전용면적", "층", "건물연령"]):
        return "Housing structure"
    if any(x in clean for x in ["계약연도", "계약월"]):
        return "Contract time"
    if any(x in clean for x in ["초등학교", "중학교", "고등학교", "초중고"]):
        return "School"
    if any(x in clean for x in ["공원"]):
        return "Park"
    if any(x in clean for x in ["바다"]):
        return "Sea"
    return "Other"


best_group = "B_location_spatial"
best_bundle = fitted[best_group]
pipe = best_bundle["estimator"]
pre = pipe.named_steps["pre"]
model = pipe.named_steps["model"]

X_for_shap = df[FEATURE_GROUPS[best_group]].copy()
sample_n = min(800, len(X_for_shap))
X_sample = X_for_shap.sample(sample_n, random_state=RND)
X_transformed = pre.transform(X_sample)
feature_names = pre.get_feature_names_out()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_transformed)
mean_abs = np.abs(shap_values).mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean |SHAP|": mean_abs,
})
shap_importance["Feature group"] = shap_importance["Feature"].map(classify_feature_group)
shap_importance = shap_importance.sort_values("Mean |SHAP|", ascending=False)
shap_importance.to_csv(OUT_DIR / "slide_28_shap_feature_importance.csv", index=False, encoding="utf-8-sig")

group_shap = (
    shap_importance.groupby("Feature group", as_index=False)["Mean |SHAP|"]
    .sum()
    .sort_values("Mean |SHAP|", ascending=False)
)
group_shap["SHAP share (%)"] = group_shap["Mean |SHAP|"] / group_shap["Mean |SHAP|"].sum() * 100
group_shap.to_csv(OUT_DIR / "slide_27_shap_group_importance.csv", index=False, encoding="utf-8-sig")

display(group_shap)
display(shap_importance.head(20))


In [ ]:
group_shap_display = group_shap.copy()
group_shap_display["Mean |SHAP|"] = group_shap_display["Mean |SHAP|"].map(lambda x: f"{x:,.2f}")
group_shap_display["SHAP share (%)"] = group_shap_display["SHAP share (%)"].map(lambda x: f"{x:.2f}%")
save_table(
    group_shap_display,
    "slide_27_shap_group_table.png",
    title="SHAP Feature Group Contribution",
    subtitle="Housing structure is the dominant predictor; spatial variables add local context.",
    col_width=30,
    font_size=11,
    scale_y=1.35
)

plt.figure(figsize=(9, 5.5))
plot_group = group_shap.sort_values("SHAP share (%)", ascending=True)
plt.barh(plot_group["Feature group"], plot_group["SHAP share (%)"], color=PALETTE["blue"])
plt.title("SHAP Share by Feature Group", fontsize=17, fontweight="bold", color=PALETTE["navy"])
plt.xlabel("Share of total mean |SHAP| (%)")
for i, v in enumerate(plot_group["SHAP share (%)"]):
    plt.text(v + 0.4, i, f"{v:.2f}%", va="center", fontsize=10, weight="bold")
save_current_fig("slide_27_shap_group_bar.png")

top_features = shap_importance.head(18).copy()
top_features_display = top_features[["Feature", "Feature group", "Mean |SHAP|"]].copy()
top_features_display["Mean |SHAP|"] = top_features_display["Mean |SHAP|"].map(lambda x: f"{x:,.2f}")
save_table(
    top_features_display,
    "slide_28_shap_top_features_table.png",
    title="Top SHAP Features",
    subtitle="The top features combine housing structure and selected local-context variables.",
    col_width=36,
    font_size=9.5,
    scale_y=1.15
)

plt.figure(figsize=(9, 7))
plot_top = shap_importance.head(18).iloc[::-1]
plt.barh(plot_top["Feature"], plot_top["Mean |SHAP|"], color=PALETTE["green"])
plt.title("Top SHAP Features", fontsize=17, fontweight="bold", color=PALETTE["navy"])
plt.xlabel("Mean |SHAP|")
save_current_fig("slide_28_shap_top_features_bar.png")

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_transformed, feature_names=feature_names, max_display=18, show=False)
plt.title("SHAP Summary Plot: B_location_spatial", fontsize=15, fontweight="bold", color=PALETTE["navy"])
save_current_fig("slide_28_shap_summary_plot.png")


## 10. Discussion, Limitations, and Conclusion Assets

These table-style assets can be inserted into the final discussion slides.


In [ ]:
findings_table = pd.DataFrame([
    ["Prediction", "XGBoost reached R² = 0.8977 and MAPE = 11.39% in the spatial model."],
    ["Random-split gain", "Spatial variables improved every metric, but the gain was small."],
    ["Holdout validation", "The spatial model improved time, apartment-complex, and dong holdout results."],
    ["SHAP", "Housing structure dominated prediction; school and park variables added local context."],
    ["Final interpretation", "Spatial variables support generalization rather than replacing housing attributes."],
], columns=["Point", "Interpretation"])
save_table(
    findings_table,
    "slide_29_main_findings_table.png",
    title="Main Findings",
    subtitle="Use careful wording: local-context support, not large random-split improvement.",
    col_width=55,
    font_size=10.5,
    scale_y=1.35
)

limitations_table = pd.DataFrame([
    ["Fixed conversion rate", "Monthly-rent rows were converted using a fixed 6.5% rate."],
    ["Dong-level spatial variables", "Variables do not measure exact apartment-to-facility distance."],
    ["Simplified sea variable", "Sea proximity is a dummy, not a view or accessibility measure."],
    ["Interpretability", "SHAP shows model contribution, not causal effect."],
    ["Regional scope", "Results are limited to Buk-gu, Pohang."],
], columns=["Limitation", "Explanation"])
save_table(
    limitations_table,
    "slide_30_limitations_table.png",
    title="Limitations",
    subtitle="These limitations should be stated directly in the final presentation.",
    col_width=48,
    font_size=10.5,
    scale_y=1.35
)

conclusion_table = pd.DataFrame([
    ["Conclusion 1", "Housing structure variables are the main predictors of jeonse-equivalent deposit."],
    ["Conclusion 2", "Spatial-derived variables add small random-split gains after dong is controlled."],
    ["Conclusion 3", "Spatial variables improve generalization in stricter holdout settings."],
    ["Final takeaway", "Spatial variables provide additional local context beyond dong-level location."],
], columns=["Conclusion", "Statement"])
save_table(
    conclusion_table,
    "slide_31_conclusion_table.png",
    title="Conclusion",
    subtitle="Final claim: spatial variables add local context and support generalization.",
    col_width=55,
    font_size=10.5,
    scale_y=1.35
)


## 11. Asset Manifest and Download

The following manifest tells the PPT team which file to insert into each slide.


In [ ]:
manifest_rows = [
    [6, "Final Dataset", "slide_06_dataset_overview_table.png", "Dataset overview table"],
    [8, "Dataset Snapshot", "slide_08_dataset_metric_cards.png", "Four large metric cards"],
    [8, "Transaction Composition", "slide_08_transaction_composition.png", "Jeonse vs converted monthly-rent chart"],
    [9, "Target Distribution", "slide_09_target_distribution.png", "Target distribution with mean and median"],
    [10, "Target Formula", "slide_10_target_formula_table.png", "Target construction formula table"],
    [12, "Spatial Variables", "slide_12_spatial_variable_categories_table.png", "Spatial variable category table"],
    [13, "Dong Harmonization", "slide_13_dong_harmonization_table.png", "Dong-level consistency check"],
    [14, "Feature Sets", "slide_14_feature_set_design_table.png", "A/B feature set table"],
    [15, "VIF Check", "slide_15_vif_redundancy_check_table.png", "Redundancy check table"],
    [16, "Feature Decisions", "slide_16_feature_selection_decisions_table.png", "Final inclusion/exclusion decisions"],
    [18, "Dong-Level EDA", "slide_18_dong_mean_target_bar.png", "Average target by dong"],
    [18, "Dong-Level Correlation", "slide_18_dong_correlation_heatmap.png", "Dong-level correlation heatmap"],
    [18, "Modeling Pipeline", "slide_18_modeling_pipeline.png", "Pipeline diagram"],
    [18, "Model Implementation", "slide_18_model_implementation_table.png", "Model settings table"],
    [19, "Evaluation Metrics", "slide_19_evaluation_metrics_table.png", "Metric explanation table"],
    [20, "Main Results", "slide_20_main_result_table.png", "Main result table"],
    [20, "Main Metrics", "slide_20_main_metrics_comparison.png", "Metric comparison bar charts"],
    [21, "Prediction vs Actual", "slide_21_prediction_vs_actual.png", "A/B prediction scatter plots"],
    [22, "Main Delta", "slide_22_main_delta_table.png", "Main performance change table"],
    [24, "Validation Design", "slide_24_validation_design_table.png", "Holdout design explanation"],
    [25, "Holdout Delta", "slide_25_holdout_delta_table.png", "Holdout improvement table"],
    [25, "Holdout Bars", "slide_25_holdout_delta_bars.png", "Holdout improvement bar charts"],
    [27, "SHAP Group Table", "slide_27_shap_group_table.png", "SHAP group contribution table"],
    [27, "SHAP Group Bar", "slide_27_shap_group_bar.png", "SHAP share by group"],
    [28, "SHAP Top Features", "slide_28_shap_top_features_table.png", "Top SHAP features table"],
    [28, "SHAP Top Feature Bar", "slide_28_shap_top_features_bar.png", "Top SHAP features bar chart"],
    [28, "SHAP Summary", "slide_28_shap_summary_plot.png", "SHAP summary plot"],
    [29, "Main Findings", "slide_29_main_findings_table.png", "Discussion findings table"],
    [30, "Limitations", "slide_30_limitations_table.png", "Limitations table"],
    [31, "Conclusion", "slide_31_conclusion_table.png", "Conclusion table"],
]
asset_manifest = pd.DataFrame(manifest_rows, columns=["Slide", "Section", "File", "Use"])
asset_manifest.to_csv(OUT_DIR / "asset_manifest.csv", index=False, encoding="utf-8-sig")
display(asset_manifest)

save_table(
    asset_manifest,
    "asset_manifest_table.png",
    title="PPT Asset Manifest",
    subtitle="Insert the listed PNG files into the corresponding slides.",
    col_width=32,
    font_size=8.7,
    scale_y=1.05
)

readme_lines = ["# PPT Asset Manifest", ""]
for _, row in asset_manifest.iterrows():
    readme_lines.append(f"- Slide {row['Slide']} | {row['Section']}: `{row['File']}` - {row['Use']}")
(OUT_DIR / "README_ppt_assets.md").write_text("\n".join(readme_lines), encoding="utf-8")

zip_path = shutil.make_archive("ppt_assets", "zip", OUT_DIR)
print("Created ZIP:", zip_path)
print("All assets are saved in:", OUT_DIR.resolve())


In [ ]:
# Optional: uncomment this cell in Colab to download every generated asset as one ZIP file.
# from google.colab import files
# files.download("ppt_assets.zip")
